# 04 — Modèles PyTorch : LSTM et CNN 1D

Fenêtres glissantes de 30 cycles. Validation : moteurs tenus à part (`GroupShuffleSplit`), jamais des lignes.
Suivi des expériences avec MLflow (fonctionne tel quel sur Databricks).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
SUBSET = "FD001"   # puis "FD004"


In [ ]:
import torch
from sklearn.model_selection import GroupShuffleSplit

from src.data import prepare
from src.features import make_sequences, last_sequences
from src.models import build_model, train_model, predict, predict_mc_dropout, set_seed, get_device
from src.evaluate import evaluate, error_by_rul_bucket, results_table

CAP, WINDOW = 125, 30
set_seed(42)
device = get_device(); print("appareil :", device)

train, test, rul_test, sensors = prepare(SUBSET, cap=CAP)
X, y, groups = make_sequences(train, sensors, WINDOW)
X_test, test_units = last_sequences(test, sensors, WINDOW)
y_test = rul_test.loc[test_units].clip(upper=CAP).to_numpy()

tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups))
assert set(groups[tr_idx]).isdisjoint(groups[va_idx]), "fuite : un moteur est des deux côtés"
print("fenêtres entraînement / validation :", len(tr_idx), len(va_idx))

In [ ]:
try:
    import mlflow
    mlflow.set_experiment(f"cmapss-rul-{SUBSET}")
except Exception as exc:  # MLflow absent ou mal configuré : on continue sans suivi
    print("Suivi MLflow désactivé :", exc.__class__.__name__)
    mlflow = None

CONFIGS = {
    "LSTM (PyTorch)": dict(name="lstm", hidden=64, layers=2, dropout=0.2),
    "CNN 1D (PyTorch)": dict(name="cnn", channels=32, kernel=5, dropout=0.2),
}
rows, nets, histories = [], {}, {}
for label, cfg in CONFIGS.items():
    set_seed(42)
    cfg = dict(cfg); name = cfg.pop("name")
    net = build_model(name, X.shape[-1], **cfg)
    hist = train_model(net, X[tr_idx], y[tr_idx], X[va_idx], y[va_idx], epochs=80, lr=1e-3, device=device)
    pred = np.clip(predict(net, X_test), 0, None)
    row = {**evaluate(y_test, pred, label), "temps (s)": hist["seconds"]}
    rows.append(row); nets[label] = net; histories[label] = hist
    if mlflow:
        with mlflow.start_run(run_name=f"{name}-{SUBSET}"):
            mlflow.log_params({"model": name, "subset": SUBSET, "cap": CAP, "window": WINDOW, **cfg})
            mlflow.log_metrics({"rmse_test": row["RMSE"], "nasa_score_test": row["score NASA"],
                                "rmse_val_best": min(hist["val_rmse"])})
res = results_table(rows)
res.to_csv(ROOT / "reports" / f"resultats_pytorch_{SUBSET}.csv")
res

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, h in histories.items():
    ax.plot(h["val_rmse"], label=f"{label} — validation")
ax.set(xlabel="époque", ylabel="RMSE", title="Courbes d'apprentissage"); ax.legend()
plt.tight_layout(); plt.show()

## RUL prédite vs réelle (graphique principal du README)

In [ ]:
label = res["RMSE"].idxmin()
pred = np.clip(predict(nets[label], X_test), 0, None)
order = np.argsort(y_test)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(y_test[order], "k-", lw=2, label="RUL réelle")
ax.plot(pred[order], "o", ms=4, alpha=0.7, label=f"RUL prédite — {label}")
ax.set(xlabel="moteurs de test (triés par RUL réelle)", ylabel="RUL (cycles)",
       title=f"{SUBSET} — RMSE {res.loc[label, 'RMSE']:.1f} cycles"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / f"rul_predite_vs_reelle_{SUBSET}.png", dpi=150); plt.show()
error_by_rul_bucket(y_test, pred)

## Option avancée : intervalle d'incertitude par MC Dropout

Un gestionnaire préfère « entre 15 et 25 cycles » à « 20 cycles ».

In [ ]:
mean, lo, hi = predict_mc_dropout(nets[label], X_test, n_samples=50)
coverage = np.mean((y_test >= lo) & (y_test <= hi))
print(f"Couverture de l'intervalle 90 % : {coverage:.0%}  ·  largeur moyenne : {np.mean(hi - lo):.1f} cycles")
pd.DataFrame({"réel": y_test, "prédit": mean.round(1), "bas (5 %)": lo.round(1), "haut (95 %)": hi.round(1)},
             index=pd.Index(test_units, name="moteur")).head(10)

In [ ]:
torch.save(nets[label].state_dict(), ROOT / "reports" / f"meilleur_modele_{SUBSET}.pt")